# Stage 05b: Hyperparameter Optimization

**Purpose:** Find optimal XGBoost hyperparameters using actuarial lift metrics

**Inputs:**
- data/04c_train_encoded.parquet
- data/04c_test_encoded.parquet
- data/04b_train.parquet (for target/exposure)
- data/04b_test.parquet
- config_generated/04c_monotonicity_constraints.yaml

**Outputs:**
- results/05b_best_params.yaml (HPO-tuned parameters only)
- results/05b_hpo_results.csv (all trial results)
- results/05b_metrics.yaml (best fit_quality, model_power, score)

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_liab/v1"


In [3]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import xgboost as xgb
import yaml, os, sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error
from itertools import product

sys.path.insert(0, str(Path.cwd() / "lib"))
from utils import setup_notebook_environment
from hpo_metrics import calculate_model_metrics, calculate_lift_opt_score

print("#" * 40)
print("# STAGE 05b: HYPERPARAMETER OPTIMIZATION")
print("#" * 40)

project_root = setup_notebook_environment()

########################################
# STAGE 05b: HYPERPARAMETER OPTIMIZATION
########################################


In [4]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")

Python: 3.11.15 (main, Jun 11 2026, 15:14:57) [Clang 20.1.8 ]
XGBoost: 3.1.1


In [5]:
# Get machine config
pc_num = open("current.pc").read().strip()
pc_id = f"PC{pc_num}"

config_file = f"{config_path}/config.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

output_base = cfg["machines"][pc_id]["paths"]["output_path"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]

print(f"\nOutput: {output_base}")
print(f"Target: {target}")
print(f"Exposure: {exposure}")


Output: output/car_liab/v1
Target: pp_bi
Exposure: ee_bi_imps


In [6]:
# Load encoded features (already filtered by exclusions in 04c)
print(f"\n* Loading encoded features...")
X_train = pd.read_parquet(f"{output_base}/data/04c_train_encoded.parquet")
X_test = pd.read_parquet(f"{output_base}/data/04c_test_encoded.parquet")

# Load original data for target/exposure
train_orig = pd.read_parquet(f"{output_base}/data/04b_train.parquet")
test_orig = pd.read_parquet(f"{output_base}/data/04b_test.parquet")

y_train = train_orig[target]
y_test = test_orig[target]
w_train = train_orig[exposure]
w_test = test_orig[exposure]

print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")
print(f"  Features: {X_train.shape[1]}")


* Loading encoded features...


  Train: (7481727, 198)
  Test: (7483698, 198)
  Features: 198


In [7]:
# Load monotonicity constraints
mono_file = f"{output_base}/config_generated/04c_monotonicity_constraints.yaml"
if os.path.exists(mono_file):
    print(f"\n* Loading monotonicity constraints...")
    with open(mono_file, "r") as f:
        mono_dict = yaml.safe_load(f)
    
    # Build constraints tuple (default to 0 if feature not in dict)
    monotone_constraints = tuple(mono_dict.get(col, 0) for col in X_train.columns)
    
    # Check for missing features
    missing = [col for col in X_train.columns if col not in mono_dict]
    if missing:
        print(f"  Warning: {len(missing)} features not in constraints (defaulting to 0)")
        print(f"  First few: {missing[:5]}")
    
    print(f"  Loaded {len(monotone_constraints)} constraints")
else:
    monotone_constraints = None
    print(f"\n* No monotonicity constraints found")


* Loading monotonicity constraints...
  First few: ['veh_use_commute_ind', 'veh_use_business_ind', 'veh_use_farm_ind', 'multi_pol_yes_cal', 'multi_pol_unknown_cal']
  Loaded 198 constraints


In [8]:
# Load HPO configuration
method = cfg.get('hyperparameter_optimization', {}).get('method', 'grid')
param_grid = cfg.get('hyperparameter_optimization', {}).get('param_grid')
scoring_config = cfg.get('hyperparameter_optimization', {}).get('scoring', {})

print(f"\n* HPO Method: {method}")
print(f"  Scoring config: {scoring_config}")

if method != 'grid':
    raise NotImplementedError(f'Method {method} not yet implemented. Use method=grid in config.')

# Prepare base XGBoost params
base_xgb_params = cfg['xgboost'].copy()
base_xgb_params.pop('n_estimators', None)

# Fix eval_metric for tweedie
if 'eval_metric' in base_xgb_params and 'tweedie' in base_xgb_params['eval_metric']:
    if '@' not in base_xgb_params['eval_metric']:
        variance_power = base_xgb_params.get('tweedie_variance_power', 1.5)
        base_xgb_params['eval_metric'] = f"{base_xgb_params['eval_metric']}@{variance_power}"

# Get n_estimators
mode = cfg['run_mode']
n_estimators = cfg[mode].get('hpo_trials', 100)

# Run grid search
from hpo_runner import run_grid_search

print(f"\n* Running Grid Search...")
hpo_results = run_grid_search(
    X_train, y_train, w_train,
    X_test, y_test, w_test,
    param_grid, base_xgb_params, n_estimators,
    monotone_constraints, exposure,
    scoring_config
)

best_params = hpo_results['best_params']
best_metrics = hpo_results['best_metrics']
results_df = hpo_results['results_df']

print(f"\nBest hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

print(f"\nBest metrics:")
for k, v in best_metrics.items():
    print(f"  {k}: {v:.4f}")


* HPO Method: grid
  Scoring config: {'bins': 10, 'fit_threshold': 0.7, 'steepness': 20.0, 'power_weight': 0.25, 'power_norm_cap': 0.5}

* Running Grid Search...
  Grid size: 144 combinations
  n_estimators: 20


  [1/144] New best: score=0.1762 (fit=0.6462, power=0.1021)


  [2/144] New best: score=0.5599 (fit=0.7437, power=0.1112)


  [13/144] New best: score=0.8142 (fit=0.8122, power=0.2182)


  [25/144] New best: score=0.9249 (fit=0.8523, power=0.3141)


  [26/144] New best: score=0.9987 (fit=0.8965, power=0.3342)



* Grid search complete
  Best score: 0.9987

Best hyperparameters:
  max_depth: 3
  learning_rate: 0.1
  min_child_weight: 500
  subsample: 0.8
  colsample_bytree: 1.0
  n_estimators: 20

Best metrics:
  fit_quality: 0.8965
  model_power: 0.3342
  lift_opt_score: 0.9987
  mae: 283.5725


In [9]:
# Save HPO results
from hpo_runner import save_hpo_results

save_hpo_results(output_base, best_params, best_metrics, results_df)


* Saving results...
  Best params: output/car_liab/v1/results/05b_best_params.yaml
  Metrics: output/car_liab/v1/results/05b_metrics.yaml
  All results: output/car_liab/v1/results/05b_hpo_results.csv

[OK] HPO results saved
